## Exercise 4: Traffic_Manipulated: Data manipulation and cleaning exercise (script / jupyter / marimo)

1. Load the provided traffic_manipulated.csv data set from ILIAS and print the first 2 rows with together with the
column names. (-> df = pd.read_csv("./traffic_manipulated.csv", index_col='index'))

2. Answer these questions (without having a look at the .csv-file):
- How many NaNs are contained in the entire dataframe?
- What are the mean and max values of each numeric column?
- Based on the column names and the first 2 rows, how many numeric columns would you expect? (check with dtypes, fix the
problematic row/column entry and recalculate the mean and max values for all numeric columns)
- Do all seven week-days appear in the column "Day"?
  
- Recreate a specific view (visitors_sum, Clicks_sum as column headers)
  
Overview of Table:

-> Total sum per countries for columns 'Visitors' and 'Clicks'

-> Sorted by the amount of how many valid entries the respective
countries have in 'Clicks' column

In [2]:
# load dataset 

import pandas as pd
import numpy as np

df = pd.read_csv("./data/traffic_manipulated.csv", index_col="index")

df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: './data/traffic_manipulated.csv'

In [3]:
#error is simple directory issue, check where python is running from 

import os
print("CWD:", os.getcwd())
print("Here:", os.listdir())

CWD: /Users/elenafuchs/advanced-python-study/lecture notes
Here: ['webscraper.ipynb', 'data manipulation and cleaning_exercise.ipynb', '.ipynb_checkpoints', 'starwars_api.ipynb', 'data_manipulation_traffic_manipulated.ipynb']


In [5]:
#correct path for data folder one level above: 

# load dataset 

import pandas as pd
import numpy as np

df = pd.read_csv("../data/traffic_manipulated.csv", index_col="index")

df.head(2)

,Visitors,Country,Clicks,Day
index,,,,
1,1500.0,USA,20,Tuesday
2,900.0,India,37,Thursday


In [6]:
#How many NaNs in entire DataFrame?

df.isna().sum().sum()

4

In [7]:
#per column

df.isna().sum()

Visitors    2
Country     0
Clicks      0
Day         2
dtype: int64

In [8]:
# Mean and Max of Each Numeric Column -> check datatypes + 

df.dtypes
df.select_dtypes(include=np.number).agg(["mean", "max"])

,Visitors
mean,4203.125
max,22100.000


### How Many Numeric Columns Should We Expect?

2 numeric columns are expected, because you see: 

Visitors → numeric

Clicks → numeric

Country → categorical

Day → categorical

In [9]:
# verify 

df.select_dtypes(include=np.number).columns

Index(['Visitors'], dtype='object')

In [10]:
# visitors is partly stored as object, change to numeric

df["Visitors"] = pd.to_numeric(df["Visitors"], errors="coerce")
df["Clicks"] = pd.to_numeric(df["Clicks"], errors="coerce")

In [11]:
# Do All Seven Weekdays Appear?

df["Day"].unique()

array(['Tuesday', 'Thursday', nan, 'Friday', 'Saturday', 'Sunday',
       'Monday', 'Wednesday'], dtype=object)

### Observation 

All seven weekdays appear in the “Day” column; however, one missing value (NaN) was detected and removed to ensure consistency of categorical data.

In [12]:
# Inspect: How Many Missing Days?

df["Day"].isna().sum()

2

### Options

1. drop row 
2. Replace Missing with Placeholder

In [15]:
df["Day"] = df["Day"].fillna("Unknown")

In [16]:
# Final Day Check Missing Days

df["Day"].value_counts()

Day
Thursday     2
Unknown      2
Tuesday      1
Friday       1
Saturday     1
Sunday       1
Monday       1
Wednesday    1
Name: count, dtype: int64

### Conclusion

All seven weekdays are present in the “Day” column. Two missing values were identified and replaced with the category “Unknown” to preserve the dataset’s row count and maintain categorical consistency.

In [17]:
#Final Check Overall

df.isna().sum().sum()

3

In [18]:
# there are still 3 missing values somewhere else. 
# Find Where the Remaining NaNs Are:

df.isna().sum()

Visitors    2
Country     0
Clicks      1
Day         0
dtype: int64

### Correct Treatment Missing Traffic Data

- there are missing values in visitors and clicks. 
- traffic data = count -> Counts missing in traffic data are typically interpreted as: 0

#### Conclusion

Dropping rows would distort country totals.
So filling with 0 is the correct practical decision.

In [20]:
# Clean Fix

df["Visitors"] = df["Visitors"].fillna(0)
df["Clicks"] = df["Clicks"].fillna(0)

In [21]:
#verify

df.isna().sum().sum()

0

### Summary why filling with 0 is correct here: 

- It preserves all rows

- It keeps aggregation consistent

- It matches domain logic (no visitors/clicks recorded = 0)

- It avoids artificial data inflation (like mean imputation would)

In [22]:
# Create a “valid clicks count” column

# create saved copy to reintegrate the NaN. 

import pandas as pd
import numpy as np

# Load fresh so we can count valid Clicks properly (NaNs still present)
df_raw = pd.read_csv("../data/traffic_manipulated.csv", index_col="index")

# Count valid (non-NaN) Clicks per country
valid_clicks = df_raw.groupby("Country")["Clicks"].count()

In [23]:
# Use cleaned df (or clean again quickly) to compute sums

df = df_raw.copy()

# Day: fill missing with "Unknown" (your choice B)
df["Day"] = df["Day"].fillna("Unknown")

# Numeric: convert + fill missing with 0 for aggregation
df["Visitors"] = pd.to_numeric(df["Visitors"], errors="coerce").fillna(0)
df["Clicks"] = pd.to_numeric(df["Clicks"], errors="coerce").fillna(0)

In [26]:
# Build the summary table

summary = df.groupby("Country").agg(
    Visitors_sum=("Visitors", "sum"),
    Clicks_sum=("Clicks", "sum")
)

# Sort by number of valid (non-NaN) click entries (from raw data)
summary = summary.loc[valid_clicks.sort_values(ascending=False).index]

summary

# format change in 1 decimal

summary.round(1)

,Visitors_sum,Clicks_sum
Country,,
USA,2400.0,124.0
India,1600.0,106.0
Australia,22100.0,57.0
Canada,500.0,35.0
UK,7025.0,0.0
